# oscNext L4 — İşleme, Feature Engineering ve Training Hazırlığı

Bu notebook `oscNext_L4_variables.py` + `process_L4.py` ile birlikte çalışır.
Uçtan uca akış:

```
L3 .i3  ──process_L4.py──▶  L4 .hdf5  ──bu notebook──▶  L4_*_training.parquet
```

| Bölüm | Ne yapılıyor |
|---|---|
| 0 | Konfigürasyon — anahtar isimleri `oscNext_L4_variables.py`'den okunur |
| 1 | **İşleme** — `process_L4.py` komutlarını üret, job listesi çıkar |
| 2 | **Booking doğrulaması** — üretilen HDF5'te hangi tablolar var? |
| 3 | Feature registry — script'in ürettiği isimlerle hizalı |
| 4 | Yükleme |
| 5 | Sağlık kontrolü |
| 6 | **Yeniden yazılan değişkenlerin doğrulaması** (VICH, accumulated_time, first_hlc) |
| 7 | Livetime ve ağırlıklar |
| 8 | Türetilmiş değişkenler |
| 9 | Data/MC uyumu |
| 10 | Korelasyon + feature importance |
| 11 | Train/test split ve export |

> **Referans:** oscNext technical note v00.07 — bölüm 3.4–3.6, Tablo 10–12, 18.

## 0. Konfigürasyon

Anahtar isimleri `oscNext_L4_variables.py`'den **AST ile** okunuyor (import
edilmiyor, çünkü o dosya icetray'e bağımlı ve bu notebook icetray olmayan bir
ortamda da çalışmalı). Böylece script'te bir anahtarı değiştirirsen notebook
otomatik olarak takip eder — iki yerin birbirinden kopması engellenir.

In [ ]:
import os, re, ast, json, glob, shlex, subprocess, warnings
from dataclasses import dataclass
from typing import Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

# ---------------------------------------------------------------------------
# YOLLAR
# ---------------------------------------------------------------------------
# Scriptlerin yeri (oscNext_L4_variables.py, process_L4.py bu dizinde)
L4_CODE_DIR = os.environ.get("OSCNEXT_L4_CODE", ".")

# --- CIKTI YOLLARI ------------------------------------------------------
# Her sey scriptlerin yanindaki "L4_output/" klasorune yazilir:
#
#   <script dizini>/L4_output/
#       hdf5/        islenmis L4 dosyalari (kind/sample alt klasorleri)
#       i3/          --output-i3 kullanilirsa
#       training/    parquet + meta (notebook cikti)
#       models/      egitilmis .txt + .json modeller
#
# DIKKAT: HDF5 ciktisi BUYUK olabilir.  Script dizininiz kucuk kotali bir
# home dizinindeyse OUTPUT_ROOT'u /data/user/$USER/... gibi bir yere alin.
# Girdi L3 yollari bolum 1'de SAMPLES icinde (zaten dolu).

OUTPUT_ROOT = os.environ.get(
    "OSCNEXT_OUT_ROOT", os.path.join(os.path.abspath(L4_CODE_DIR), "L4_output"))

HDF_BASE  = os.path.join(OUTPUT_ROOT, "hdf5")
I3_BASE   = os.path.join(OUTPUT_ROOT, "i3")
OUT       = os.path.join(OUTPUT_ROOT, "training")
MODEL_DIR = os.path.join(OUTPUT_ROOT, "models")

for _d in (HDF_BASE, OUT, MODEL_DIR):
    try:
        os.makedirs(_d, exist_ok=True)
    except OSError as e:
        print(f"[!] {_d} olusturulamadi: {e}")
        print("    OUTPUT_ROOT'u yazma izniniz olan bir yere degistirin.")

# Diskte ne kadar yer var?
try:
    _st = os.statvfs(OUTPUT_ROOT)
    _free_gb = _st.f_bavail * _st.f_frsize / 1e9
    print(f"Bos disk ({OUTPUT_ROOT}): {_free_gb:,.1f} GB")
    if _free_gb < 20:
        print("  [!] 20 GB'tan az bos yer var -- HDF5 ciktisi icin yetmeyebilir.")
except OSError:
    pass

VARS_PY    = os.path.join(L4_CODE_DIR, "oscNext_L4_variables.py")
PROCESS_PY = os.path.join(L4_CODE_DIR, "process_L4.py")

RNG_SEED = 12345
rng = np.random.default_rng(RNG_SEED)


# ---------------------------------------------------------------------------
# Anahtar isimlerini AST ile cek (import etmeden)
# ---------------------------------------------------------------------------
def load_constants(path):
    """
    Modul seviyesindeki sabitleri, kodu CALISTIRMADAN cek.

    Sadece literal, isim referansi ve +/% islemlerini cozer -- yani
    L4_FIRST_HLC_RHO_KEY = L4_FIRST_HLC_KEY + "_rho"  ve
    MICROCOUNT_SUBKEY    = "STW_m%ip%i_DTW%i" % (STW_MINUS, STW_PLUS, DTW)
    gibi turetilmis sabitler de dogru cozulur.
    """
    if not os.path.exists(path):
        print(f"[!] {path} bulunamadi -- varsayilan isimler kullanilacak")
        return {}
    tree = ast.parse(open(path).read())
    ns = {}

    def ev(node):
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.Name):
            if node.id in ns:
                return ns[node.id]
            raise ValueError(node.id)
        if isinstance(node, (ast.Tuple, ast.List)):
            return tuple(ev(e) for e in node.elts)
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return -ev(node.operand)
        if isinstance(node, ast.BinOp):
            l, r = ev(node.left), ev(node.right)
            if isinstance(node.op, ast.Add):  return l + r
            if isinstance(node.op, ast.Mod):  return l % r
            if isinstance(node.op, ast.Mult): return l * r
        raise ValueError(ast.dump(node)[:40])

    for node in tree.body:
        if not isinstance(node, ast.Assign) or len(node.targets) != 1:
            continue
        tgt = node.targets[0]
        if not isinstance(tgt, ast.Name):
            continue
        try:
            ns[tgt.id] = ev(node.value)     # yeniden atama varsa sonuncusu kazanir
        except (ValueError, TypeError):
            pass
    return ns

K = load_constants(VARS_PY)

def key(name, default):
    v = K.get(name, default)
    return v

# Frame objesi isimleri (script ile senkron)
KEY_FIRST_HLC     = key("L4_FIRST_HLC_KEY",      "L4_first_hlc")
KEY_FIRST_HLC_RHO = key("L4_FIRST_HLC_RHO_KEY",  "L4_first_hlc_rho")
KEY_TOI           = key("L4_TOI_KEY",            "L4_ToI")
KEY_LINEFIT       = key("L4_LINEFIT_KEY",        "L4_iLineFit")
KEY_QRBOX         = key("L4_QRBOX_KEY",          "L4_QR_Box")
KEY_VICH_NCH      = key("L4_VICH_NCH_KEY",       "L4_VICH_nch")
KEY_VICH_NPULSES  = key("L4_VICH_NPULSES_KEY",   "L4_VICH_npulses")
KEY_VICH_QTOT     = key("L4_VICH_QTOT_KEY",      "L4_VICH_qtot")
KEY_SEP_COG       = key("L4_SEP_IN_COGS_KEY",    "L4_separation_in_cogs")
KEY_ACC_TIME      = key("L4_ACC_TIME_KEY",       "L4_accumulated_time")
KEY_MICROCOUNT    = key("L4_MICROCOUNT_KEY",     "L4_micro_count")
KEY_FILL_RATIO    = key("L4_FILL_RATIO_KEY",     "L4_fill_ratio")
KEY_HITSTAT       = key("HITSTAT_KEY",           "SRTTWOfflinePulsesDCHitStatistics")
KEY_HITMULT       = key("HITMULT_KEY",           "SRTTWOfflinePulsesDCHitMultiplicity")
KEY_NOISE_PROB    = key("L4_NOISE_MODEL_PREDICTION_KEY",     "L4_NoiseClassifier_ProbNu")
KEY_MUON_PROB     = key("L4_MUON_MODEL_PREDICTION_DATA_KEY", "L4_MuonClassifier_Data_ProbNu")

MICRO_SUBKEY = key("MICROCOUNT_SUBKEY", "STW_m3500p4000_DTW200")
VICH_SPEED   = (key("VICH_SPEED_MIN", 0.25), key("VICH_SPEED_MAX", 0.40))
FILL_RADIUS  = key("FILL_RATIO_SPHERICAL_RADIUS_MEAN", 1.6)

print("Script'ten okunan anahtarlar:")
for n, v in [("micro_count subkey", MICRO_SUBKEY), ("VICH hiz penceresi", VICH_SPEED),
             ("fill ratio yaricap", FILL_RADIUS), ("hit statistics", KEY_HITSTAT)]:
    print(f"  {n:22s} {v}")

# L4 kesim degerleri (v00.07)
NOISE_CUT, MUON_CUT = 0.70, 0.65

## 1. İşleme — `process_L4.py` çalıştırma

Burada iş listesini üretiyoruz. Küçük testleri notebook'tan doğrudan
çalıştırabilirsin; tam üretim için komutları dosyaya yazıp cluster'a gönder.

**Örnek tanımları.** `kind` alanı hangi MC anahtarlarının book edileceğini
belirler. Gürültü MC'sinde `MCInIcePrimary` **yoktur** — o yüzden `--noise`
bayrağı ayrı.

Muon BDT'sinin arka planı için run seçimi kritik: yılları eşit kapsayacak
şekilde seçilmeli, yoksa mevsimsel muon akı değişimi sınıflandırıcıya sızar.

In [ ]:
# ---------------------------------------------------------------------------
# GERCEK pass3 YOLLARI  (mevcut oscnext_rates.py PATHS'inden)
#
#   NuE      23800
#   NuMu     23799
#   NuTau    YOK        -> sinyal = nue + numu.  Kayip kucuk (nutau CC L3'te
#                          ~0.129 mHz, toplam sinyalin ~%3'u) ama nutau'ya
#                          ozgu bir davranis varsa siniflandirici onu gormez.
#   CORSIKA  23694      -> atmosferik muon (MuonGun DEGIL)
#   Noise    23813      -> vuvuzela
#   Data     YOK        -> bkz. MUON_BACKGROUND asagida
#
# HANGI ORNEK NEREYE GIDIYOR:
#
#   ornek     noise BDT    muon BDT     ayrica
#   --------  -----------  -----------  --------------------
#   nue       SINYAL       SINYAL       data/MC
#   numu      SINYAL       SINYAL       data/MC
#   noise     ARKA PLAN    --           data/MC
#   corsika   --           ARKA PLAN    data/MC + VICH dogrulama
#
# Noise BDT'de muon YOK: sadece vuvuzela + GENIE ile egitilir.
# Muon BDT arka plani normalde GERCEK VERI olurdu; veri olmadigi icin
# CORSIKA kullaniliyor (bkz. MUON_BACKGROUND).
# ---------------------------------------------------------------------------

OSC_L3 = "/data/ana/LE/oscNext/pass3"

SAMPLES = {
    "nue": dict(
        kind="genie", sets=[23800],
        l3_glob=[f"{OSC_L3}/genie/level3/23800/genie_NuE_IC86.023800.*.i3.zst"]),
    "numu": dict(
        kind="genie", sets=[23799],
        l3_glob=[f"{OSC_L3}/genie/level3/23799/genie_NuMu_IC86.023799.*.i3.zst"]),
    "corsika": dict(
        kind="corsika", sets=[23694],
        l3_glob=[f"{OSC_L3}/corsika/level3/23694/corsika_IC86.023694.*.i3.zst"]),
    "noise": dict(
        kind="noise", sets=[23813],
        l3_glob=[f"{OSC_L3}/noise/level3/23813/noise_IC86.023813.*.i3.zst"]),
    # --- detektor verisi: yolu bulursaniz doldurun ---
    # "data": dict(kind="data", sets=None,
    #              l3_glob=[f"{OSC_L3}/data/level3/Run*/*.i3.zst"]),
}

DEFAULT_GCD = ("/cvmfs/icecube.opensciencegrid.org/data/GCD/"
               "GeoCalibDetectorStatus_2020.Run134142.Pass2_V0.i3.gz")

for _n, _c in SAMPLES.items():
    _c.setdefault("gcd", None if _c["kind"] == "data" else DEFAULT_GCD)
    _c["hdf_dir"] = f"{HDF_BASE}/{_c['kind']}/{_n}"
    _c["hdf_glob"] = f"{_c['hdf_dir']}/*.hdf5"

# ---------------------------------------------------------------------------
# MUON BDT ARKA PLANI
#
#   "data"    : oscNext'in yaklasimi.  Bu asamada veri %99 muon ve MuonGun
#               MC'den daha guvenilir sayiliyor; ayrica simule edilmemis
#               populasyonlari (muon bundle) da temizliyor.
#   "corsika" : veri yoksa.  Dokuman MuonGun ile egitilmis bir varyantin
#               BENZER performans elde ettigini soyluyor, yani MC-egitimli
#               yaklasim calisiyor.  CORSIKA MuonGun'dan daha iyi bile
#               olabilir -- muon bundle'lari iceriyor.
#
# Veri bulursaniz: SAMPLES'a "data" ekleyip burayi "data" yapin.
# ---------------------------------------------------------------------------
MUON_BACKGROUND = "data" if "data" in SAMPLES else "corsika"
print(f"Muon BDT arka plani: {MUON_BACKGROUND}")

ROLE = {}
for _n, _c in SAMPLES.items():
    if _c["kind"] == "genie":
        ROLE[_n] = dict(noise="signal", muon="signal")
    elif _c["kind"] == "noise":
        ROLE[_n] = dict(noise="background", muon=None)
    elif _n == MUON_BACKGROUND:
        ROLE[_n] = dict(noise=None, muon="background")
    else:
        ROLE[_n] = dict(noise=None, muon=None)

_roles = pd.DataFrame(ROLE).T.fillna("-").rename(
    columns={"noise": "noise BDT", "muon": "muon BDT"})
_roles["set"] = [str(SAMPLES[i].get("sets") or "-") for i in _roles.index]
display(_roles)

# Muon BDT arka plani gercek veriyse: yillari dengeli kapsayan run'lar
# (mevsimsel muon akisi degisiminin siniflandiriciya sizmasini onler)
MUON_TRAINING_RUNS = []

# --- Girdi dosyalari gercekten var mi? ---
print()
_tot = 0
for _n, _c in SAMPLES.items():
    _files = sorted({f for p in _c["l3_glob"] for f in glob.glob(p)})
    _tot += len(_files)
    _flag = "[OK] " if _files else "[YOK]"
    print(f"  {_flag} {_n:9s} {len(_files):6d} L3 dosyasi")
    if not _files:
        print(f"         -> {_c['l3_glob'][0]}")
if _tot == 0:
    print("\n[!] Hicbir L3 dosyasi bulunamadi -- SAMPLES icindeki yollari kontrol edin.")

# Cluster'da kac L3 dosyasi tek job'da islensin
FILES_PER_JOB = 10

In [ ]:
def data_gcd_for(path):
    """Data dosyasindan run numarasini cikarip GCD yolunu kur."""
    m = re.search(r"Run(\d{8})", path) or re.search(r"Run(\d+)", path)
    if not m:
        return None
    run = int(m.group(1))
    return f"/data/GCD/Run{run:08d}_GCD.i3.zst"   # <- kendi GCD deseninize gore


def build_jobs(samples=SAMPLES, files_per_job=FILES_PER_JOB, write_i3=False,
               apply_cut=False, model_dir=None, extra_args=""):
    """process_L4.py komut listesi uret."""
    jobs = []
    for name, cfg in samples.items():
        patterns = cfg["l3_glob"]
        if isinstance(patterns, str):
            patterns = [patterns]
        files = sorted({f for p in patterns for f in glob.glob(p)})
        if not files:
            print(f"[atlandi] {name}: L3 dosyasi yok -> {patterns}")
            continue
        os.makedirs(cfg["hdf_dir"], exist_ok=True)

        # data icin run bazli grupla (her run'in kendi GCD'si var)
        if cfg["kind"] == "data":
            groups = {}
            for f in files:
                groups.setdefault(data_gcd_for(f), []).append(f)
        else:
            groups = {cfg["gcd"]: files}

        for gcd, flist in groups.items():
            if gcd is None:
                print(f"[!] {name}: GCD cozulemedi, atlandi"); continue
            for i in range(0, len(flist), files_per_job):
                chunk = flist[i:i + files_per_job]
                tag = f"{name}_{i//files_per_job:05d}"
                out_h5 = f"{cfg['hdf_dir']}/L4_{tag}.hdf5"

                cmd = ["python", PROCESS_PY, "--gcd", gcd,
                       "--input"] + chunk + ["--output-hdf5", out_h5]
                if write_i3:
                    os.makedirs(f"{I3_BASE}/{cfg['kind']}/{name}", exist_ok=True)
                    cmd += ["--output-i3", f"{I3_BASE}/{cfg['kind']}/{name}/L4_{tag}.i3.zst"]
                if cfg["kind"] == "noise":
                    cmd += ["--noise"]
                elif cfg["kind"] == "genie":
                    # --genie: I3GenieInfo.n_flux_events her frame'e tasinir
                    cmd += ["--mc", "--genie"]
                elif cfg["kind"] == "corsika":
                    # CorsikaWeightMap + PolyplopiaPrimary book edilir
                    cmd += ["--mc", "--corsika"]
                elif cfg["kind"] == "muongun":
                    cmd += ["--mc", "--muongun"]
                if apply_cut:
                    cmd += ["--apply-cut", "--model-dir", model_dir]
                if extra_args:
                    cmd += shlex.split(extra_args)

                jobs.append(dict(sample=name, kind=cfg["kind"], tag=tag,
                                 n_input=len(chunk), output=out_h5,
                                 cmd=" ".join(shlex.quote(c) for c in cmd)))
    return pd.DataFrame(jobs)


jobs = build_jobs()
if len(jobs):
    print(f"{len(jobs)} job, {jobs.n_input.sum()} L3 dosyasi")
    display(jobs.groupby(["sample", "kind"]).agg(n_jobs=("tag", "size"),
                                                 n_files=("n_input", "sum")))
    joblist = os.path.join(OUT, "L4_jobs.txt")
    with open(joblist, "w") as fh:
        fh.write("\n".join(jobs.cmd) + "\n")
    print("Komut listesi:", joblist)
    print("\nOrnek komut:\n ", jobs.cmd.iloc[0][:400])

### Job'ları notebook'tan çalıştırma

`run_jobs()` iş listesini yerel bir process havuzunda çalıştırır. Cluster
kullanacaksanız `L4_jobs.txt` dosyasını submit edin ve bu hücreyi atlayın.

Her job bir alt süreçte `process_L4.py` çağırır, yani biri patlarsa diğerleri
devam eder — hangilerinin başarısız olduğu sonunda raporlanır.

In [ ]:
import concurrent.futures as _cf

def _run_one(job):
    import subprocess, time
    t0 = time.time()
    r = subprocess.run(job["cmd"], shell=True, capture_output=True, text=True)
    return dict(tag=job["tag"], sample=job["sample"], rc=r.returncode,
                dt=time.time() - t0, output=job["output"],
                stdout=r.stdout[-2500:], stderr=r.stderr[-2500:])


def run_jobs(jobs_df, n_workers=8, skip_existing=True, dry_run=False):
    """
    Job listesini paralel calistir.

    skip_existing=True: cikti hdf5 zaten varsa atlar -> yarim kalan bir
        uretimi bastan baslatmadan tamamlayabilirsiniz.
    """
    todo = jobs_df.to_dict("records")
    if skip_existing:
        before = len(todo)
        todo = [j for j in todo if not os.path.exists(j["output"])]
        if before != len(todo):
            print(f"{before - len(todo)} job atlandi (cikti zaten var)")
    if not todo:
        print("Yapilacak job yok."); return pd.DataFrame()
    if dry_run:
        print(f"{len(todo)} job calistirilacakti. Ilk komut:\n  {todo[0]['cmd'][:300]}")
        return pd.DataFrame(todo)

    print(f"{len(todo)} job, {n_workers} isci ...")
    results, done = [], 0
    with _cf.ProcessPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(_run_one, j): j for j in todo}
        for fut in _cf.as_completed(futs):
            r = fut.result(); results.append(r); done += 1
            flag = "ok " if r["rc"] == 0 else "HATA"
            print(f"  [{done:4d}/{len(todo)}] {flag} {r['tag']:24s} {r['dt']:6.1f}s")

    res = pd.DataFrame(results)
    n_bad = int((res.rc != 0).sum())
    print(f"\nBitti: {len(res)-n_bad} basarili, {n_bad} hatali")
    if n_bad:
        print("\n--- ILK HATANIN CIKTISI ---")
        bad = res[res.rc != 0].iloc[0]
        print("tag:", bad["tag"])
        print(bad["stderr"][-2000:] or bad["stdout"][-2000:])
    return res


# --- calistir ---
#
# ADIM 1: HER ORNEKTEN birkac job.  jobs.head(N) YETMEZ -- liste sirali
# oldugu icin hepsi ayni ornekten gelir ve corsika/noise kod yollari hic
# denenmemis olur.  Bunlar farkli bayraklar ve farkli agirlik anahtarlari
# kullaniyor, dolayisiyla ayri ayri patlayabilirler.
#
#     probe = jobs.groupby("sample").head(2)
#     results = run_jobs(probe, n_workers=4)
#
# ADIM 2: hepsi gectikten sonra tam uretim
#
#     results = run_jobs(jobs, n_workers=8)

probe = jobs.groupby("sample").head(2) if len(jobs) else None
if probe is not None:
    print("Deneme job'lari:")
    display(probe[["sample", "kind", "tag", "n_input"]])
    print("\nCalistirmak icin:  results = run_jobs(probe, n_workers=4)")

results = None

### İlk önce küçük bir test

Tam üretime geçmeden önce tek dosyada 200 frame işleyip zincirin çalıştığını
doğrulayın. `--apply-cut` **kullanmayın** — sınıflandırıcılar henüz eğitilmedi.

In [ ]:
def run_smoke_test(sample="nue", n_frames=200, timeout=1800):
    cfg = SAMPLES[sample]
    patterns = cfg["l3_glob"] if isinstance(cfg["l3_glob"], list) else [cfg["l3_glob"]]
    files = sorted({f for p in patterns for f in glob.glob(p)})
    if not files:
        print("L3 dosyasi yok, test atlandi"); return None
    gcd = cfg["gcd"] or data_gcd_for(files[0])
    out = os.path.join(OUT, f"smoketest_{sample}.hdf5")

    cmd = ["python", PROCESS_PY, "--gcd", gcd, "--input", files[0],
           "--output-hdf5", out, "--n", str(n_frames)]
    if cfg["kind"] == "noise":
        cmd += ["--noise"]
    elif cfg["kind"] == "genie":
        cmd += ["--mc", "--genie"]
    elif cfg["kind"] == "corsika":
        cmd += ["--mc", "--corsika"]
    elif cfg["kind"] == "muongun":
        cmd += ["--mc", "--muongun"]

    print(" ".join(cmd), "\n")
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print("--- STDERR ---\n", r.stderr[-4000:])
        return None
    return out

# smoke_h5 = run_smoke_test("nue")     # <- ortaminizda calistirin
smoke_h5 = None

## 2. Booking doğrulaması

İşleme bittikten sonra **ilk iş bu**: HDF5'te beklenen tabloların hepsi var mı,
satır sayıları tutarlı mı? Bir modül sessizce başarısız olduysa tablo hiç
yazılmaz ya da çok az satır içerir — bunu ileride NaN olarak görmek yerine
burada yakalayın.

`--sub-event-stream` yanlışsa hiçbir frame işlenmez ve **sessizce boş HDF5**
alırsınız; aşağıdaki kontrol onu da yakalar.

In [ ]:
import tables

IDX = ["Run", "Event", "SubEvent"]

EXPECTED_TABLES = {
    # tablo adi -> hangi orneklerde beklenir (None = hepsinde)
    "I3EventHeader":            None,
    "IC2018_LE_L3_Vars":        None,
    KEY_HITSTAT:                None,
    KEY_HITMULT:                None,
    KEY_FIRST_HLC:              None,
    KEY_FIRST_HLC_RHO:          None,
    KEY_MICROCOUNT:             None,
    KEY_FILL_RATIO:             None,
    KEY_LINEFIT + "Params":     None,
    KEY_TOI + "Params":         None,
    KEY_VICH_NCH:               None,
    KEY_VICH_NPULSES:           None,
    KEY_VICH_QTOT:              None,
    KEY_ACC_TIME:               None,
    KEY_SEP_COG:                None,
    "I3MCWeightDict":           {"genie", "muongun"},
    "L4_n_flux_events":         {"genie"},
    "noise_weight":             {"noise"},
    "CorsikaWeightMap":         {"corsika"},
    "PolyplopiaPrimary":        {"corsika"},
}
# NOT: pass3'te MCInIcePrimary YOK -- truth I3MCWeightDict icinde
# (PrimaryNeutrinoEnergy / Zenith / Type), o yuzden beklenenler listesinde degil.


def inspect_file(path, pattern=None):
    rows = []
    with tables.open_file(path, "r") as h5:
        for node in h5.walk_nodes("/", "Table"):
            if pattern and not re.search(pattern, node.name, re.I):
                continue
            cols = [c for c in node.colnames
                    if c not in IDX + ["SubEventStream", "exists"]]
            rows.append(dict(table=node.name, nrows=node.nrows,
                             ncols=len(cols), columns=", ".join(cols[:10])))
    return pd.DataFrame(rows).sort_values("table").reset_index(drop=True)


def verify_booking(path, kind):
    """Beklenen tablolar var mi, satir sayilari tutarli mi?"""
    with tables.open_file(path, "r") as h5:
        present = {n.name: n.nrows for n in h5.walk_nodes("/", "Table")}
    if not present:
        return pd.DataFrame([dict(table="<HIC TABLO YOK>", status="FAIL",
                                  nrows=0, note="sub-event-stream yanlis olabilir")])
    ref = present.get("I3EventHeader", max(present.values()))
    rows = []
    for t, kinds in EXPECTED_TABLES.items():
        if kinds is not None and kind not in kinds:
            continue
        if t not in present:
            rows.append(dict(table=t, status="MISSING", nrows=0, frac=0.0)); continue
        n = present[t]
        frac = n / max(ref, 1)
        status = "OK" if frac > 0.95 else ("PARTIAL" if frac > 0.05 else "NEARLY_EMPTY")
        rows.append(dict(table=t, status=status, nrows=n, frac=round(frac, 4)))
    return pd.DataFrame(rows)


if smoke_h5:
    display(inspect_file(smoke_h5))
    print("\n--- Booking dogrulamasi ---")
    display(verify_booking(smoke_h5, SAMPLES["nue"]["kind"]))

In [ ]:
# Uretim ciktisi uzerinde toplu kontrol: her ornekten bir dosya orneklenir
def verify_all(samples=SAMPLES, per_sample=1):
    out = []
    for name, cfg in samples.items():
        files = sorted(glob.glob(cfg["hdf_glob"]))
        if not files:
            print(f"[{name}] HDF5 yok -> {cfg['hdf_glob']}"); continue
        for f in files[:per_sample]:
            v = verify_booking(f, cfg["kind"])
            v["sample"] = name
            out.append(v)
    if not out:
        return pd.DataFrame()
    df = pd.concat(out, ignore_index=True)
    return df.pivot_table(index="table", columns="sample", values="status",
                          aggfunc="first")

vt = verify_all()
if len(vt):
    display(vt)

## 3. Feature registry

Her değişken `(tablo, kolon)` çifti. `process_L4.py`'nin ürettiği yapıya göre:

| Frame objesi | HDF5 kolonu |
|---|---|
| `I3Double` (VICH, accumulated_time, rho) | `value` |
| `I3MapStringDouble/Int` (L3 vars, micro_count) | her map anahtarı bir kolon |
| `I3LineFitParams` | `LFVel`, `LFVelX/Y/Z`, `NHits` |
| `I3HitStatisticsValues` | `cog_x/y/z`, `z_min`, `z_max`, `z_sigma`, `z_travel`, … |
| `I3FillRatioInfo` | `fill_ratio_from_mean`, `fill_radius_from_mean`, … |
| `I3Particle` (first_hlc) | `x`, `y`, `z`, `time`, `zenith`, … |

`alts` alanı isim varyasyonlarını çözer — `resolve_features()` dosyaya bakıp
hangisinin gerçekten var olduğunu bulur, böylece meta-proje sürümü değişse
bile registry'yi elle düzeltmek gerekmez.

In [ ]:
@dataclass
class Feature:
    name: str
    table: str
    column: str
    alts: Sequence[tuple] = ()
    valid_range: tuple = (-np.inf, np.inf)
    note: str = ""

L3V = "IC2018_LE_L3_Vars"

# --- NOISE BDT girdileri (Tablo 11) ---
NOISE_FEATURES = [
    Feature("NchCleaned", L3V, "NchCleaned", valid_range=(0, 5000),
            note="Temizlenmis seride hit alan DOM sayisi"),
    Feature("micro_count", KEY_MICROCOUNT, MICRO_SUBKEY,
            alts=[(KEY_MICROCOUNT, "STW7500_DTW200")], valid_range=(0, 5000),
            note="[-3.5,+4] us icinde 200 ns pencerede maks DOM sayisi"),
    # DOGRULANDI: pass3 hdfwriter kolonu "lf_vel" (eski varsayim "LFVel" DEGIL)
    Feature("iLineFit_speed", KEY_LINEFIT + "Params", "lf_vel",
            alts=[(KEY_LINEFIT + "Params", "LFVel"), (KEY_LINEFIT, "speed")],
            valid_range=(0, 10), note="improved LineFit hizi [m/ns]"),
    Feature("fill_ratio", KEY_FILL_RATIO, "fill_ratio_from_mean",
            alts=[(KEY_FILL_RATIO, "fillratio_from_mean"),
                  (KEY_FILL_RATIO, "fill_ratio_from_mean_plus_rms"),
                  (KEY_FILL_RATIO, "fillratio_from_mean_plus_rms")],
            valid_range=(0, 1),
            note="Vertex etrafindaki kure icinde hit alan DOM fraksiyonu"),
    # pass3 L3 map'inde oran YOK (Cleaned/UncleanedFullTimeLength ayri ayri var)
    # -> L4 tray'i L4_FullTimeLengthRatio olarak hesaplayip yaziyor
    Feature("FullTimeLengthRatio", "L4_FullTimeLengthRatio", "value",
            alts=[(L3V, "FullTimeLengthRatio")], valid_range=(0, 100),
            note="Temizlenmis / temizlenmemis olay suresi orani"),
]

# --- MUON BDT girdileri (Tablo 12) ---
MUON_FEATURES = [
    Feature("ICVetoHits",    L3V, "ICVetoHits",    valid_range=(0, 5000)),
    Feature("RTVeto250Hits", L3V, "RTVeto250Hits", valid_range=(0, 5000)),
    Feature("NchCleaned",    L3V, "NchCleaned",    valid_range=(0, 5000)),
    Feature("NAbove200Hits", L3V, "NAbove200Hits", valid_range=(0, 5000)),
    Feature("VICH_nch", KEY_VICH_NCH, "value", valid_range=(0, 5000),
            note="Muondan kaynaklanabilecek veto bolgesi DOM sayisi"),
    Feature("accumulated_time", KEY_ACC_TIME, "value", valid_range=(0, 2e4),
            note="Yukun %75'ine ulasma suresi [ns]"),
    Feature("first_hlc_rho", KEY_FIRST_HLC_RHO, "value", valid_range=(0, 2000),
            note="Ilk HLC hit'in string 36'ya radyal uzakligi [m]"),
    Feature("cog_z", KEY_HITSTAT, "cog_z", valid_range=(-1000, 1000)),
    # I3HitStatisticsValues'ta z yayilimi bazi surumlerde "cog_z_sigma"
    # olarak tabulasyona giriyor -- ikisini de dene
    Feature("z_sigma", KEY_HITSTAT, "z_sigma",
            alts=[(KEY_HITSTAT, "cog_z_sigma")], valid_range=(0, 2000)),
    Feature("z_travel", KEY_HITSTAT, "z_travel", valid_range=(-2000, 2000),
            note="ISARETLI -- mutlak deger almayin"),
]

# --- Aday degiskenler: hesaplaniyor ama v00.07'de kullanilmamis ---
CANDIDATE_FEATURES = [
    Feature("ToI_evalratio", KEY_TOI + "Params", "evalratio", valid_range=(0, 1)),
    Feature("ToI_mineval", KEY_TOI + "Params", "mineval"),
    Feature("QR_Box", KEY_QRBOX, "value"),
    Feature("separation_in_cogs", KEY_SEP_COG, "value", valid_range=(0, 2000),
            note="TANIMI DOGRULANMADI -- bkz. bolum 6"),
    Feature("VICH_npulses", KEY_VICH_NPULSES, "value", valid_range=(0, 1e5)),
    Feature("VICH_qtot",    KEY_VICH_QTOT,    "value", valid_range=(0, 1e6),
            note="YUKE BAGLI -- SPE modelleme hatalarina duyarli"),
    Feature("first_hlc_x", KEY_FIRST_HLC, "x", valid_range=(-1000, 1000)),
    Feature("first_hlc_y", KEY_FIRST_HLC, "y", valid_range=(-1000, 1000)),
    Feature("first_hlc_z", KEY_FIRST_HLC, "z", valid_range=(-1000, 1000)),
    Feature("first_hlc_time", KEY_FIRST_HLC, "time"),
    Feature("iLineFit_nhits", KEY_LINEFIT + "Params", "n_hits",
            alts=[(KEY_LINEFIT + "Params", "NHits")], valid_range=(0, 5000)),
    Feature("CausalVetoHits", L3V, "CausalVetoHits", valid_range=(0, 5000)),
    Feature("C2HR6", L3V, "C2HR6", valid_range=(0, 1)),
    Feature("VetoFiducialRatioHits", L3V, "VetoFiducialRatioHits"),
    Feature("VertexGuessZ", L3V, "VertexGuessZ", valid_range=(-1000, 1000)),
    Feature("DCFiducialHits", L3V, "DCFiducialHits", valid_range=(0, 5000)),
    Feature("STW9000_DTW300Hits", L3V, "STW9000_DTW300Hits", valid_range=(0, 5000)),
    Feature("CleanedFullTimeLength", L3V, "CleanedFullTimeLength", valid_range=(0, 1e5)),
    Feature("UncleanedFullTimeLength", L3V, "UncleanedFullTimeLength", valid_range=(0, 1e5)),
    Feature("n_hit_doms", KEY_HITMULT, "n_hit_doms", valid_range=(0, 5000)),
    Feature("cog_x", KEY_HITSTAT, "cog_x", valid_range=(-1000, 1000)),
    Feature("cog_y", KEY_HITSTAT, "cog_y", valid_range=(-1000, 1000)),
    Feature("z_min", KEY_HITSTAT, "z_min", valid_range=(-1000, 1000)),
    Feature("z_max", KEY_HITSTAT, "z_max", valid_range=(-1000, 1000)),
]

ALL_FEATURES = {}
for f in NOISE_FEATURES + MUON_FEATURES + CANDIDATE_FEATURES:
    ALL_FEATURES.setdefault(f.name, f)
FEATURE_LIST = list(ALL_FEATURES.values())

print(f"{len(NOISE_FEATURES)} noise + {len(MUON_FEATURES)} muon + "
      f"{len(CANDIDATE_FEATURES)} aday -> {len(ALL_FEATURES)} benzersiz")

## 4. Yükleme

Tablolar `Run/Event/SubEvent` üzerinden **merge** ediliyor, satır sırasına
güvenilmiyor. Bir frame objesi bazı olaylarda yoksa satır kayması olur ve
sessizce yanlış değişkenler eşleşir — bu tip analizin en sinsi hatası.

Ayrıca livetime hesabı için `I3EventHeader`'ın zaman alanları da çekiliyor.

In [ ]:
def _read_table(h5, table, columns):
    try:
        node = h5.get_node("/" + table)
    except tables.NoSuchNodeError:
        return None
    have = set(node.colnames)
    want = [c for c in columns if c in have]
    if not want:
        return None
    idx = [c for c in IDX if c in have]
    arr = node.read()
    df = pd.DataFrame({c: arr[c] for c in idx + want})
    if "exists" in have:
        df = df[arr["exists"].astype(bool)]
    return df


def resolve_features(path, features, verbose=False):
    """Dosyada gercekten var olan (tablo, kolon) ciftini bul."""
    with tables.open_file(path, "r") as h5:
        available = {n.name: set(n.colnames) for n in h5.walk_nodes("/", "Table")}
    resolved, missing = {}, []
    for f in features:
        hit = None
        for (t, c) in [(f.table, f.column)] + list(f.alts):
            if t in available and c in available[t]:
                hit = (t, c); break
        if hit: resolved[f.name] = hit
        else:   missing.append(f.name)
    if verbose and missing:
        print("  cozulemedi:", missing)
    return resolved, missing


EXTRA_HEADER = {
    "start_mjd_s": ("I3EventHeader", "time_start_mjd_sec"),
    "end_mjd_s":   ("I3EventHeader", "time_end_mjd_sec"),
    "start_mjd_d": ("I3EventHeader", "time_start_mjd_day"),
}
# DIKKAT: pass3 I3MCWeightDict'inde "weight" alani YOK (onu oscNext islemesi
# ekliyor, o proje bizde yok).  Ham alanlardan kendimiz hesaplayacagiz.
EXTRA_MC = {
    "OneWeight":     ("I3MCWeightDict", "OneWeight"),
    "NEvents":       ("I3MCWeightDict", "NEvents"),
    "n_flux_events": ("L4_n_flux_events", "value"),   # tray tarafindan tasindi
}
EXTRA_NOISE   = {"noise_weight_raw": ("noise_weight", "weight")}
EXTRA_MUONGUN = {"mg_weight": ("I3MCWeightDict", "weight")}
# CORSIKA: agirlik simweights ile hesaplanir, ham alanlar lazim
EXTRA_CORSIKA = {
    "cwm_Weight":        ("CorsikaWeightMap", "Weight"),
    "cwm_NEvents":       ("CorsikaWeightMap", "NEvents"),
    "cwm_OverSampling":  ("CorsikaWeightMap", "OverSampling"),
    "cwm_PrimaryEnergy": ("CorsikaWeightMap", "PrimaryEnergy"),
    "cwm_PrimarySpectralIndex": ("CorsikaWeightMap", "PrimarySpectralIndex"),
    "cwm_EnergyPrimaryMin":     ("CorsikaWeightMap", "EnergyPrimaryMin"),
    "cwm_EnergyPrimaryMax":     ("CorsikaWeightMap", "EnergyPrimaryMax"),
    "cwm_CylinderLength":       ("CorsikaWeightMap", "CylinderLength"),
    "cwm_CylinderRadius":       ("CorsikaWeightMap", "CylinderRadius"),
    "cwm_ThetaMin":             ("CorsikaWeightMap", "ThetaMin"),
    "cwm_ThetaMax":             ("CorsikaWeightMap", "ThetaMax"),
    "pp_energy": ("PolyplopiaPrimary", "energy"),
    "pp_zenith": ("PolyplopiaPrimary", "zenith"),
    "pp_type":   ("PolyplopiaPrimary", "type"),
}
# pass3'te MCInIcePrimary yok -> truth I3MCWeightDict icinden
EXTRA_TRUTH = {
    "true_energy": ("I3MCWeightDict", "PrimaryNeutrinoEnergy"),
    "true_zenith": ("I3MCWeightDict", "PrimaryNeutrinoZenith"),
    "pdg":         ("I3MCWeightDict", "PrimaryNeutrinoType"),
    "int_type":    ("I3MCWeightDict", "InteractionType"),
}


def load_sample(file_list, features, extra_tables=None, max_files=None, verbose=True):
    extra_tables = dict(extra_tables or {})
    if max_files: file_list = file_list[:max_files]

    resolved, missing = resolve_features(file_list[0], features, verbose=verbose)
    plan = {}
    for name, (t, c) in resolved.items():
        plan.setdefault(t, {})[c] = name
    for name, (t, c) in extra_tables.items():
        plan.setdefault(t, {})[c] = name

    frames, n_ok = [], 0
    for i, path in enumerate(file_list):
        merged = None
        try:
            with tables.open_file(path, "r") as h5:
                for t, colmap in plan.items():
                    df = _read_table(h5, t, list(colmap.keys()))
                    if df is None: continue
                    df = df.rename(columns=colmap)
                    on = [c for c in IDX if c in df.columns]
                    df = df.drop_duplicates(subset=on)
                    merged = df if merged is None else merged.merge(df, on=on, how="outer")
        except Exception as e:
            if verbose: print(f"  [!] {os.path.basename(path)}: {e}")
            continue
        if merged is not None and len(merged):
            merged["__file__"] = i
            frames.append(merged); n_ok += 1

    if not frames:
        raise RuntimeError("Hicbir dosyadan veri okunamadi.")
    out = pd.concat(frames, ignore_index=True)
    out.attrs["n_files"] = n_ok
    if verbose:
        print(f"  {n_ok} dosya, {len(out):,} olay, {out.shape[1]} kolon")
    return out

In [ ]:
MAX_FILES = None      # gelistirme sirasinda kucuk bir sayi verin (orn. 20)

samples = {}
for name, cfg in SAMPLES.items():
    files = sorted(glob.glob(cfg["hdf_glob"]))
    if not files:
        print(f"[atlandi] {name}: {cfg['hdf_glob']}"); continue
    print(f"[{name}] ({cfg['kind']}) {len(files)} hdf5")

    extra = dict(EXTRA_HEADER)
    if cfg["kind"] == "genie":
        extra.update(EXTRA_MC); extra.update(EXTRA_TRUTH)
    elif cfg["kind"] == "noise":
        extra.update(EXTRA_NOISE)
    elif cfg["kind"] == "corsika":
        extra.update(EXTRA_CORSIKA)
    elif cfg["kind"] == "muongun":
        extra.update(EXTRA_MUONGUN)

    df = load_sample(files, FEATURE_LIST, extra_tables=extra, max_files=MAX_FILES)
    df["sample"], df["kind"] = name, cfg["kind"]
    samples[name] = df

print("\nYuklendi:", {k: len(v) for k, v in samples.items()})

## 5. Sağlık kontrolü

Her örnek × değişken için: coverage, inf, aralık dışı, sabitlik.
LightGBM NaN'ı doğal olarak işler ama `inf` patlatır.

Bilinen bir sorun: `HitStatistics` MuonGun dosyalarının ~%0.1'inde L2'de
başarısız oluyor; nedeni bilinmiyor ve bu olaylar L3 kesimiyle temizleniyor.
Yani `cog_z`/`z_sigma`/`z_travel` için çok küçük bir kayıp normal.

In [ ]:
def health_report(samples, features):
    rows = []
    for sname, df in samples.items():
        for f in features:
            if f.name not in df.columns:
                rows.append(dict(sample=sname, feature=f.name, status="MISSING",
                                 coverage=0.0, n_inf=0, n_oor=0, n_unique=0, median=np.nan))
                continue
            x = pd.to_numeric(df[f.name], errors="coerce").to_numpy(float)
            n, finite = len(x), np.isfinite(x)
            n_inf = int(np.isinf(x).sum())
            lo, hi = f.valid_range
            n_oor = int((finite & ((x < lo) | (x > hi))).sum())
            cov = float(finite.sum()) / max(n, 1)
            v = x[finite]
            status = "OK"
            if   cov < 0.5:            status = "LOW_COVERAGE"
            elif n_inf > 0:            status = "HAS_INF"
            elif n_oor > 0.001 * n:    status = "OUT_OF_RANGE"
            elif v.size and np.unique(v).size == 1: status = "CONSTANT"
            rows.append(dict(sample=sname, feature=f.name, status=status,
                             coverage=round(cov, 4), n_inf=n_inf, n_oor=n_oor,
                             n_unique=int(np.unique(v).size) if v.size else 0,
                             median=round(float(np.median(v)), 4) if v.size else np.nan))
    return pd.DataFrame(rows)

rep = health_report(samples, FEATURE_LIST)

def gate(feature_defs, label):
    names = [f.name for f in feature_defs]
    sub = rep[rep.feature.isin(names)]
    print(f"\n=== {label} ===")
    display(sub.pivot(index="feature", columns="sample", values="status"))
    print("SONUC:", "hepsi OK" if (sub.status == "OK").all() else ">>> duzeltilmesi gereken var")

gate(NOISE_FEATURES, "Noise BDT girdileri (Tablo 11)")
gate(MUON_FEATURES,  "Muon BDT girdileri (Tablo 12)")

In [ ]:
problems = rep[rep.status != "OK"].sort_values(["feature", "sample"])
display(problems if len(problems) else "Tum degiskenler saglikli.")

## 6. Yeniden yazılan değişkenlerin doğrulaması

`oscNext_L4_variables.py` şu dört şeyi saf Python'la yeniden yazıyor, çünkü
orijinal projeler (`tau_bdt`, `analysis.event_selection`, `slc-veto`,
`FirstHLC` C++ modülü) modern IceTray'de yok:

| Değişken | Risk | Belirsizlik |
|---|---|---|
| `VICH_nch` | **Yüksek** | `dt` yönü (`t_COG − t_hit > 0` alındı), veto DOM tanımı |
| `accumulated_time` | Orta | Referans zamanı, fraksiyon eşiği |
| `first_hlc_rho` | Düşük | DOM başına ilk HLC seçimi |
| `separation_in_cogs` | Yüksek ama BDT girdisi değil | Bölme kuralı |

**En iyi test:** elinizde v01.03 referans L4 dosyası varsa aynı L3 girdisini
iki koddan geçirip olay-olay karşılaştırın (`compare_to_reference`).

**Referans yoksa:** fiziksel akıl sağlığı kontrolü. VICH nötrinolarda ~0'da
yoğunlaşmalı, MuonGun'da belirgin kuyruk olmalı. İkisi ayırt edilemiyorsa
nedensellik yönü veya veto DOM tanımı yanlış demektir.

In [ ]:
def compare_to_reference(new_h5, ref_h5, features=None, n_show=8):
    """Ayni olaylari iki HDF5'ten okuyup olay-olay karsilastir."""
    features = features or [ALL_FEATURES[n] for n in
                            ["VICH_nch", "accumulated_time", "first_hlc_rho",
                             "micro_count", "separation_in_cogs"]
                            if n in ALL_FEATURES]
    a = load_sample([new_h5], features, verbose=False)
    b = load_sample([ref_h5], features, verbose=False)
    m = a.merge(b, on=IDX, suffixes=("_new", "_ref"))
    print(f"Eslesen olay: {len(m):,}")

    rows = []
    for f in features:
        cn, cr = f.name + "_new", f.name + "_ref"
        if cn not in m or cr not in m: continue
        x = pd.to_numeric(m[cn], errors="coerce")
        y = pd.to_numeric(m[cr], errors="coerce")
        ok = np.isfinite(x) & np.isfinite(y)
        if ok.sum() < 10: continue
        d = (x - y)[ok]
        rows.append(dict(feature=f.name, n=int(ok.sum()),
                         identical_frac=round(float((np.abs(d) < 1e-6).mean()), 4),
                         mean_diff=round(float(d.mean()), 4),
                         p95_absdiff=round(float(np.percentile(np.abs(d), 95)), 4),
                         corr=round(float(np.corrcoef(x[ok], y[ok])[0, 1]), 5)))
    res = pd.DataFrame(rows)
    display(res)

    fig, axes = plt.subplots(1, min(n_show, len(rows)), figsize=(3*min(n_show, len(rows)), 2.8))
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, rows):
        cn, cr = r["feature"]+"_new", r["feature"]+"_ref"
        ax.scatter(m[cr], m[cn], s=2, alpha=0.2)
        lim = [np.nanmin(m[cr]), np.nanmax(m[cr])]
        ax.plot(lim, lim, "r-", lw=0.8)
        ax.set_xlabel("referans"); ax.set_ylabel("yeni"); ax.set_title(r["feature"], fontsize=8)
    plt.tight_layout(); plt.show()
    return res

# res = compare_to_reference("/tmp/new.hdf5", "/data/oscNext/v01.03/ref.hdf5")

In [ ]:
def sanity_rewritten(feature, log=False, bins=50, xrange=None):
    """Nu vs MuonGun vs Noise ayrimi bekledigimiz yonde mi?"""
    groups = {"nu (GENIE)": [s for s in ("nue","numu","nutau") if s in samples],
              "MuonGun":    ["muongun"] if "muongun" in samples else [],
              "Noise":      ["noise"]   if "noise"   in samples else [],
              "Data":       ["data"]    if "data"    in samples else []}
    pool = []
    for ss in groups.values():
        for s in ss:
            if feature in samples[s].columns:
                pool.append(pd.to_numeric(samples[s][feature], errors="coerce").to_numpy(float))
    if not pool: print(f"{feature}: yok"); return
    allv = np.concatenate(pool); allv = allv[np.isfinite(allv)]
    if allv.size == 0: print(f"{feature}: hep NaN"); return
    lo, hi = xrange or (np.percentile(allv, 0.5), np.percentile(allv, 99.5))
    edges = np.linspace(lo, hi, bins+1)

    plt.figure(figsize=(5, 3))
    stats = []
    for label, ss in groups.items():
        vs = [pd.to_numeric(samples[s][feature], errors="coerce").to_numpy(float)
              for s in ss if feature in samples[s].columns]
        if not vs: continue
        v = np.concatenate(vs); v = v[np.isfinite(v)]
        if v.size == 0: continue
        h, _ = np.histogram(v, bins=edges, density=True)
        plt.step(0.5*(edges[1:]+edges[:-1]), h, where="mid", label=label, lw=1.2)
        stats.append(dict(group=label, n=v.size, median=round(float(np.median(v)), 3),
                          frac_zero=round(float((v == 0).mean()), 3),
                          p90=round(float(np.percentile(v, 90)), 3)))
    if log: plt.yscale("log")
    plt.xlabel(feature); plt.ylabel("normalize yogunluk"); plt.legend(fontsize=7)
    plt.title(f"{feature} — sinif ayrimi", fontsize=9); plt.show()
    display(pd.DataFrame(stats))


for f in ["VICH_nch", "accumulated_time", "first_hlc_rho", "micro_count",
          "separation_in_cogs", "iLineFit_speed"]:
    if any(f in d.columns for d in samples.values()):
        sanity_rewritten(f, log=True)

In [ ]:
# VICH icin ozel kontrol: nu vs muon ayirici gucu
MU_SAMPLE = "corsika" if "corsika" in samples else (
            "muongun" if "muongun" in samples else None)
if MU_SAMPLE and NU:
    nu = pd.concat([samples[s] for s in NU])
    mu = samples[MU_SAMPLE]
    for col in ["VICH_nch", "ICVetoHits", "first_hlc_rho"]:
        if col not in nu.columns: continue
        a = pd.to_numeric(nu[col], errors="coerce").dropna()
        b = pd.to_numeric(mu[col], errors="coerce").dropna()
        print(f"{col:18s}  nu medyan={np.median(a):8.2f}  "
              f"muon medyan={np.median(b):8.2f}  "
              f"nu frac==0={np.mean(a==0):.3f}  muon frac==0={np.mean(b==0):.3f}")
    print("\nBEKLENTI: VICH_nch ve ICVetoHits muonlarda BELIRGIN sekilde daha yuksek.")
    print("Ayrim yoksa -> _vich icindeki dt yonu veya veto DOM tanimi hatali.")

## 7. Livetime ve ağırlıklar

Üç ayrı kavram:

1. **Fiziksel rate ağırlığı [Hz]** — dağılım çizerken ve kesim performansını
   ölçerken. MC için `I3MCWeightDict["weight"] / n_files`, veri için
   `1 / livetime`. `weight` alanı o dosyaya normalize edilmiş bir hızdır,
   doğru ağırlık için kullanılan dosya sayısına bölmek gerekir.
2. **Eğitim ağırlığı** — dokümandaki ön işleme: sinyal ve arka planın toplam
   ağırlıkları eşitlenir, sonra tüm ağırlıklar sayısal hata birikimini önlemek
   için 0–1 aralığına çekilir.
3. **Ağırlıksız sayım** — istatistiksel yeterlilik için.

MC istatistiği asimetrik: nötrinolar için ~10× detektör livetime'ı, muonlar
için ~1×, gürültü için ~1 ay. Noise BDT'sinin biraz overtrained olması bilinen
sorunlar arasında ve sebebi bu.

Livetime `I3EventHeader`'dan hesaplanıyor (run başına ilk/son olay farkı).
Detector downtime'ı hesaba katmaz, ama eğitim için yeterli — ağırlıklar zaten
sonradan sınıf bazında normalize ediliyor.

In [ ]:
def compute_livetime(df):
    """Run basina (max - min) event zamani toplami [s]."""
    if "start_mjd_s" not in df.columns:
        return None
    t = pd.to_numeric(df["start_mjd_s"], errors="coerce")
    if "start_mjd_d" in df.columns:
        d = pd.to_numeric(df["start_mjd_d"], errors="coerce")
        t = d * 86400.0 + t
    tmp = pd.DataFrame({"Run": df["Run"], "t": t}).dropna()
    if tmp.empty:
        return None
    per_run = tmp.groupby("Run")["t"].agg(lambda s: s.max() - s.min())
    return float(per_run.sum()), per_run


DATA_LIVETIME_S = None
if "data" in samples:
    res = compute_livetime(samples["data"])
    if res:
        DATA_LIVETIME_S, per_run = res
        print(f"Data livetime: {DATA_LIVETIME_S:,.0f} s "
              f"({DATA_LIVETIME_S/86400:.2f} gun), {len(per_run)} run")
    else:
        print("[!] Livetime hesaplanamadi -- I3EventHeader book edilmemis olabilir.")
        DATA_LIVETIME_S = 1.0

In [ ]:
# ---------------------------------------------------------------------------
# Agirliklar -- mevcut oscnext_rates.py ile AYNI konvansiyon
#
# GENIE:   weight [Hz] = OneWeight * flux(E) / n_flux
#          flux(E)     = NORM * E**GAMMA
#          n_flux      = I3GenieInfo.n_flux_events            (varsa)
#                      = NEvents * 0.7 (nu) / 0.3 (nubar)     (yoksa)
# Noise:   noise_weight * birim carpani  (pass3 -> 1/ns, x1e9)
# MuonGun: frame'deki hazir weight
# Data:    1 / livetime
#
# NORM/GAMMA gercek atmosferik aki DEGIL, basit bir guc yasasi yaklasimi.
# Mutlak oranlar Tablo 13 ile birebir tutmaz ama kendi icinde tutarlidir ve
# data/MC sekil karsilastirmasi icin yeterlidir.  Gercek aki icin nuflux
# (Honda) + salinim gerekir.
# ---------------------------------------------------------------------------

NORM, GAMMA = 2e-2, -3.0
NU_NEVENTS_FRAC, NUBAR_NEVENTS_FRAC = 0.7, 0.3

# Vuvuzela noise_weight birimi: pass3 -> 1/ns (x1e9), pass2 -> zaten Hz
NOISE_NS_SCALE = 1.0 if REPRODUCE_V0007 else 1e9


def genie_weight(df, n_files):
    E  = pd.to_numeric(df.get("true_energy"), errors="coerce")
    ow = pd.to_numeric(df.get("OneWeight"), errors="coerce")
    flux = NORM * np.power(E, GAMMA)

    if "n_flux_events" in df.columns:
        n_flux = pd.to_numeric(df["n_flux_events"], errors="coerce")
    else:
        n_flux = pd.Series(np.nan, index=df.index)

    missing = ~np.isfinite(n_flux)
    if missing.any():
        nev = pd.to_numeric(df.get("NEvents"), errors="coerce")
        pdg = pd.to_numeric(df.get("pdg"), errors="coerce")
        frac = pd.Series(np.where(pdg < 0, NUBAR_NEVENTS_FRAC, NU_NEVENTS_FRAC),
                         index=df.index)
        n_flux = n_flux.where(~missing, nev * frac)
        print(f"  [i] {int(missing.sum()):,} olayda n_flux_events yok "
              f"-> NEvents * ({NU_NEVENTS_FRAC}/{NUBAR_NEVENTS_FRAC})")

    # n_flux dosya bazinda -> dosya sayisina bol
    return (ow * flux / n_flux) / float(n_files)


def corsika_weight(sample_name, df, n_files):
    """
    CORSIKA agirligi -- simweights + GaisserH3a (oscnext_rates.py ile ayni).

    simweights HDF5 dosyalarini dogrudan okur, o yuzden dosyalari yeniden
    aciyoruz.  Sonuc olay bazinda degil dosya-seti bazinda hizalanir; burada
    Run/Event/SubEvent uzerinden geri eslestiriyoruz.

    simweights yoksa: yaklasik guc yasasi ile devam eder ve UYARIR.
    """
    try:
        import simweights
    except ImportError:
        print("  [!] simweights YOK -> CORSIKA agirligi yaklasik "
              "(CorsikaWeightMap.Weight / NEvents).  Mutlak oran guvenilmez.")
        w = pd.to_numeric(df.get("cwm_Weight"), errors="coerce")
        nev = pd.to_numeric(df.get("cwm_NEvents"), errors="coerce")
        osamp = pd.to_numeric(df.get("cwm_OverSampling"), errors="coerce").fillna(1)
        return w / (nev * osamp * float(n_files))

    files = sorted(glob.glob(SAMPLES[sample_name]["hdf_glob"]))[:MAX_FILES or None]
    parts = []
    for f in files:
        try:
            store = pd.HDFStore(f, "r")
        except Exception as e:
            print(f"  [!] {os.path.basename(f)} simweights icin acilamadi: {e}")
            continue
        try:
            wobj = simweights.CorsikaWeighter(store, nfiles=1)
            w = wobj.get_weights(simweights.GaisserH3a())
            idx = store["/I3EventHeader"][["Run", "Event", "SubEvent"]].copy()
            idx["w_sw"] = w
            parts.append(idx)
        except Exception as e:
            print(f"  [!] simweights basarisiz ({os.path.basename(f)}): {e}")
        finally:
            store.close()

    if not parts:
        print("  [!] simweights hicbir dosyada calismadi -> NaN")
        return pd.Series(np.nan, index=df.index)

    wdf = pd.concat(parts, ignore_index=True).drop_duplicates(subset=IDX)
    merged = df[IDX].merge(wdf, on=IDX, how="left")
    print(f"  simweights: {len(wdf):,} olay agirliklandirildi "
          f"({100*merged['w_sw'].notna().mean():.0f}% eslesme)")
    return merged["w_sw"].to_numpy() / float(n_files)


for name, df in samples.items():
    kind = SAMPLES[name]["kind"]
    nf = df.attrs.get("n_files") or df["__file__"].nunique()

    if kind == "data":
        df["w_phys"] = 1.0 / (DATA_LIVETIME_S or 1.0)
    elif kind == "genie":
        df["w_phys"] = genie_weight(df, nf)
    elif kind == "noise":
        w = pd.to_numeric(df.get("noise_weight_raw"), errors="coerce")
        df["w_phys"] = w * NOISE_NS_SCALE / float(nf)
    elif kind == "corsika":
        df["w_phys"] = corsika_weight(name, df, nf)
    elif kind == "muongun":
        df["w_phys"] = pd.to_numeric(df.get("mg_weight"), errors="coerce") / float(nf)
    else:
        df["w_phys"] = np.nan

    tot = np.nansum(df["w_phys"])
    ok = float(np.isfinite(pd.to_numeric(df["w_phys"], errors="coerce")).mean())
    print(f"{name:10s} {kind:8s} N={len(df):9,d} dosya={nf:4d} "
          f"rate={tot:.4e} Hz  (gecerli agirlik: {100*ok:.0f}%)")

### Ağırlık sağlık kontrolü

Bu sayılar zincirin doğruluğunun en iyi tek göstergesi. v00.07 pass2'de L3
sonrası beklenen mertebeler (pass3'te birebir tutmaz, mertebe benzer olmalı):

| Bileşen | L3 oranı |
|---|---|
| νe CC | ~0.95 mHz |
| νμ CC | ~3.77 mHz |
| ντ CC | ~0.129 mHz |
| Atm. μ | ~505 mHz |
| Gürültü | ~36.6 mHz |

Mertebe olarak saparsa sırayla şüphelen:

1. **Dosya sayısı** — kısmi job çalıştırıldı mı, `n_files` doğru mu?
2. **`noise_weight` birimi** — pass3 1/ns, pass2 Hz. `NOISE_NS_SCALE` doğru mu?
3. **`n_flux_events`** — `[i]` uyarısı çıktıysa `--genie` bayrağı unutulmuş demektir
4. **NORM/GAMMA** — güç yasası yaklaşımı, mutlak ölçek zaten yaklaşık

In [ ]:
# Tek bir olay toplami domine etmemeli
n = len(samples)
fig, axes = plt.subplots(1, n, figsize=(3.2*n, 2.8))
axes = np.atleast_1d(axes)
for ax, (name, df) in zip(axes, samples.items()):
    w = pd.to_numeric(df["w_phys"], errors="coerce").to_numpy(float)
    w = w[np.isfinite(w) & (w > 0)]
    if w.size == 0:
        ax.set_title(f"{name}: agirlik yok", fontsize=8)
        continue
    ax.hist(np.log10(w), bins=40, color="tab:blue")
    ax.set_title(f"{name}\nmaks/toplam = {w.max()/w.sum():.1%}", fontsize=8)
    ax.set_xlabel("log10(w_phys)", fontsize=7)
    ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()

print("maks/toplam > %5 ise tek bir olay oraninizi domine ediyor:")
print("istatistik yetersiz ya da agirlik hesabinda sorun var.")

## 8. Türetilmiş değişkenler

Ham değişkenlerin üstüne fiziksel motivasyonu olan kombinasyonlar. Bunlar
**aday**dır — bölüm 9 (data/MC) ve 10 (importance) filtresinden geçmeden
nihai listeye alınmamalı.

Temel ilke korunuyor: oscNext L3 bilinçli olarak yük bağımlılığından
uzaklaştırıldı (SPE şablon modelleme hatalarına duyarlılığı azaltmak için,
yük tabanlı değişkenler DOM-sayısı analoglarıyla değiştirildi). Türetilmiş
değişkenlerde de mümkünse yük yerine DOM sayısı kullanın.

In [ ]:
STRING36 = (46.29, -34.88)

def add_derived(d):
    g = lambda c: (pd.to_numeric(d[c], errors="coerce")
                   if c in d.columns else pd.Series(np.nan, index=d.index))
    eps = 1e-9
    # muona duyarli oranlar
    d["veto_over_nch"]   = g("ICVetoHits")    / (g("NchCleaned") + eps)
    d["vich_over_nch"]   = g("VICH_nch")      / (g("NchCleaned") + eps)
    d["nabove_over_nch"] = g("NAbove200Hits") / (g("NchCleaned") + eps)
    d["rtveto_over_nch"] = g("RTVeto250Hits") / (g("NchCleaned") + eps)
    # geometri
    d["zsigma_over_ztravel"] = g("z_sigma") / (g("z_travel").abs() + eps)
    d["z_extent"]   = g("z_max") - g("z_min")
    d["cog_rho"]    = np.hypot(g("cog_x") - STRING36[0], g("cog_y") - STRING36[1])
    d["cog_z_minus_hlc_z"] = g("cog_z") - g("first_hlc_z")
    d["hlc_to_cog_dist"] = np.sqrt((g("cog_x")-g("first_hlc_x"))**2 +
                                   (g("cog_y")-g("first_hlc_y"))**2 +
                                   (g("cog_z")-g("first_hlc_z"))**2)
    # gurultuye duyarli: zaman sikismasi
    d["micro_over_nch"]   = g("micro_count") / (g("NchCleaned") + eps)
    d["micro_l3_over_l4"] = g("STW9000_DTW300Hits") / (g("micro_count") + eps)
    d["nch_per_ns"]       = g("NchCleaned") / (g("CleanedFullTimeLength") + eps)
    d["log_speed"]        = np.log10(np.clip(g("iLineFit_speed"), 1e-4, None))
    # yuke bagli -- data/MC kontrolu sart
    d["qtot_per_dom"]     = g("VICH_qtot") / (g("VICH_nch") + eps)
    return d

DERIVED_NAMES = ["veto_over_nch","vich_over_nch","nabove_over_nch","rtveto_over_nch",
                 "zsigma_over_ztravel","z_extent","cog_rho","cog_z_minus_hlc_z",
                 "hlc_to_cog_dist","micro_over_nch","micro_l3_over_l4","nch_per_ns",
                 "log_speed","qtot_per_dom"]

for k in list(samples):
    samples[k] = add_derived(samples[k])

rep_d = health_report(samples, [Feature(n, "", "") for n in DERIVED_NAMES])
display(rep_d.pivot(index="feature", columns="sample", values="coverage"))

## 9. Data/MC uyum kontrolü

**Atlanamaz.** Toplam ~40 değişken test edildi ve nihai listeye girenler iyi
data-MC uyumu *ve* güçlü feature importance gösterenler oldu; model
karmaşıklığını ve overtraining olasılığını sınırlamak için değişken sayısı
bilinçli olarak minimize edildi.

Bir değişkende uyumsuzluk varsa BDT onu öğrenir ve gerçek veride beklediğinden
farklı davranır.

Muon değişkenlerine **noise kesimi uygulanmış** halde bakılmalı. Sınıflandırıcı
henüz eğitilmediyse `after_noise_cut` düz kesimlere düşer.

In [ ]:
def after_noise_cut(df):
    """L4 noise kesimi. Model varsa onu, yoksa duz kesimleri kullanir."""
    if KEY_NOISE_PROB.replace(".", "_") in df.columns:
        return pd.to_numeric(df[KEY_NOISE_PROB], errors="coerce").to_numpy() > NOISE_CUT
    g = lambda c: (pd.to_numeric(df[c], errors="coerce").to_numpy(float)
                   if c in df.columns else np.full(len(df), np.nan))
    with np.errstate(invalid="ignore"):
        m = ((g("n_hit_doms") >= 8) & (g("STW9000_DTW300Hits") >= 2) &
             (g("micro_count") >= 2) & (g("fill_ratio") >= 0.03) &
             (g("z_sigma") >= 8.) & (g("z_travel") >= -50.))
    return np.nan_to_num(m, nan=False).astype(bool)


def plot_data_mc(feature, bins=40, log=True, cut_mask=None, xrange=None):
    labels = {"nue":"nue","numu":"numu","nutau":"nutau",
              "noise":"Noise","muongun":"MuonGun","corsika":"CORSIKA"}
    colors = {"nue":"tab:blue","numu":"tab:orange","nutau":"tab:green",
              "noise":"tab:grey","muongun":"tab:red","corsika":"tab:red"}
    def vals(df):
        v = pd.to_numeric(df[feature], errors="coerce").to_numpy(float)
        w = df["w_phys"].to_numpy(float)
        m = np.isfinite(v)
        if cut_mask is not None: m &= cut_mask(df)
        return v[m], w[m]

    pool = [vals(samples[k])[0] for k in samples if feature in samples[k]]
    if not pool: print(f"{feature}: yok"); return
    pool = np.concatenate(pool); pool = pool[np.isfinite(pool)]
    if pool.size == 0: return
    lo, hi = xrange or (np.percentile(pool, 0.5), np.percentile(pool, 99.5))
    if lo == hi: hi = lo + 1
    edges = np.linspace(lo, hi, bins+1); ctr = 0.5*(edges[1:]+edges[:-1])

    fig, (ax, axr) = plt.subplots(2, 1, figsize=(5.2, 4.3), sharex=True,
                                  gridspec_kw=dict(height_ratios=[3,1], hspace=0.05))
    total = np.zeros(bins)
    for k in labels:
        if k not in samples or feature not in samples[k].columns: continue
        v, w = vals(samples[k])
        h, _ = np.histogram(v, bins=edges, weights=w)
        total += h
        ax.step(ctr, h, where="mid", label=labels[k], color=colors[k], lw=1)
    ax.step(ctr, total, where="mid", label="Total MC", color="k", lw=1.6)

    if "data" in samples and feature in samples["data"].columns:
        v, w = vals(samples["data"])
        hd, _ = np.histogram(v, bins=edges, weights=w)
        nraw, _ = np.histogram(v, bins=edges)
        wm = w.mean() if w.size else 1.0
        ax.errorbar(ctr, hd, yerr=np.sqrt(nraw)*wm, fmt="o", ms=2.5,
                    color="crimson", label="Data")
        with np.errstate(divide="ignore", invalid="ignore"):
            ax.figure and axr.errorbar(ctr, np.where(total>0, hd/total, np.nan),
                                       fmt="o", ms=2.5, color="crimson")
    axr.axhline(1., color="k", lw=0.8); axr.axhspan(0.9, 1.1, color="tab:green", alpha=0.12)
    axr.set_ylim(0.5, 1.5); axr.set_ylabel("Data/MC", fontsize=8); axr.set_xlabel(feature)
    if log: ax.set_yscale("log")
    ax.set_ylabel("Rate [1/s]"); ax.legend(fontsize=6, ncol=2)
    plt.show()


for f in NOISE_FEATURES:
    plot_data_mc(f.name)

In [ ]:
for f in MUON_FEATURES:
    plot_data_mc(f.name, cut_mask=after_noise_cut)

In [ ]:
def data_mc_score(feature, bins=25, cut_mask=None, min_rate=1e-7):
    if "data" not in samples: return np.nan
    def vals(df):
        v = pd.to_numeric(df[feature], errors="coerce").to_numpy(float)
        w = df["w_phys"].to_numpy(float); m = np.isfinite(v)
        if cut_mask is not None: m &= cut_mask(df)
        return v[m], w[m]
    pool = [vals(samples[k])[0] for k in samples if feature in samples[k]]
    if not pool: return np.nan
    pool = np.concatenate(pool)
    if pool.size == 0: return np.nan
    edges = np.linspace(np.percentile(pool,1), np.percentile(pool,99), bins+1)
    if edges[0] == edges[-1]: return np.nan
    tot = np.zeros(bins)
    for k in samples:
        if k == "data" or feature not in samples[k].columns: continue
        v, w = vals(samples[k]); tot += np.histogram(v, bins=edges, weights=w)[0]
    v, w = vals(samples["data"])
    hd = np.histogram(v, bins=edges, weights=w)[0]
    m = tot > min_rate
    if m.sum() < 3: return np.nan
    return float(np.mean(np.abs(hd[m]/tot[m] - 1.0)))

rows = []
for f in NOISE_FEATURES:
    rows.append(dict(feature=f.name, cls="noise", mean_abs_dev=data_mc_score(f.name)))
for f in MUON_FEATURES:
    rows.append(dict(feature=f.name, cls="muon",
                     mean_abs_dev=data_mc_score(f.name, cut_mask=after_noise_cut)))
for n in DERIVED_NAMES + [f.name for f in CANDIDATE_FEATURES]:
    rows.append(dict(feature=n, cls="aday",
                     mean_abs_dev=data_mc_score(n, cut_mask=after_noise_cut)))
sc = pd.DataFrame(rows).dropna().sort_values("mean_abs_dev", ascending=False)
display(sc.head(30))
print("\nKural: mean_abs_dev > ~0.2 supheli. Elemeden once nedenini anlayin.")

## 10. Korelasyon ve feature importance

Hyperparametreler Tablo 10'daki değerler: `max_depth=6`, `num_leaves=25`,
`max_bin=32`, `min_data_in_leaf=500`, `feature_fraction` noise 0.8 / muon 0.7,
`lambda_l1=2.0`, `lambda_l2=1.0`, `min_gain_to_split=2.0`, `is_unbalanced=False`.
Noise sınıflandırıcısı %33.3 eğitim / %66.7 doğrulama ile eğitilmişti.

Üç importance metriği bakılıyor. **Permutation'a daha çok güvenin** — gain,
çok değerli sürekli değişkenler lehine biaslıdır ve `NchCleaned` gibi tamsayı
değişkenleri haksız yere aşağı çeker.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

LGB_BASE = dict(objective="binary", metric="auc", verbosity=-1,
                max_depth=6, num_leaves=25, max_bin=32, min_data_in_leaf=500,
                lambda_l1=2.0, lambda_l2=1.0, min_gain_to_split=2.0,
                is_unbalance=False, learning_rate=0.05,
                seed=RNG_SEED, deterministic=True)
LGB_NOISE = {**LGB_BASE, "feature_fraction": 0.8}
LGB_MUON  = {**LGB_BASE, "feature_fraction": 0.7}
TRAIN_FRAC_NOISE, TRAIN_FRAC_MUON = 1/3, 0.5


def build_training_set(sig_frames, bkg_frames, feature_names,
                       weight_col="w_phys", clip_quantile=0.999):
    sig = pd.concat(sig_frames, ignore_index=True); sig["label"] = 1
    bkg = pd.concat(bkg_frames, ignore_index=True); bkg["label"] = 0
    df = pd.concat([sig, bkg], ignore_index=True)
    w = pd.to_numeric(df[weight_col], errors="coerce").to_numpy(float)
    lab = df["label"].to_numpy()

    nan = ~np.isfinite(w)
    for l in (0, 1):
        m = nan & (lab == l)
        if m.any(): w[m] = 1.0
    w = np.clip(w, 0, None)
    for l in (0, 1):                       # uzun kuyrugu kirp
        m = lab == l
        if m.sum() > 10: w[m] = np.minimum(w[m], np.quantile(w[m], clip_quantile))
    for l in (0, 1):                       # sinif dengeleme
        m = lab == l
        if w[m].sum() > 0: w[m] = w[m] / w[m].sum()
    w = w / w.max()                        # [0,1]

    df["w_train"] = w
    X = df[feature_names].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    return df, X, lab, w


def quick_train(X, y, w, params, train_frac=0.5, rounds=600):
    Xtr, Xte, ytr, yte, wtr, wte = train_test_split(
        X, y, w, train_size=train_frac, random_state=RNG_SEED, stratify=y)
    d1 = lgb.Dataset(Xtr, ytr, weight=wtr, feature_name=list(X.columns))
    d2 = lgb.Dataset(Xte, yte, weight=wte, reference=d1)
    m = lgb.train(params, d1, rounds, valid_sets=[d1, d2], valid_names=["train","test"],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    a1 = roc_auc_score(ytr, m.predict(Xtr), sample_weight=wtr)
    a2 = roc_auc_score(yte, m.predict(Xte), sample_weight=wte)
    print(f"  iter={m.best_iteration}  AUC train={a1:.4f} test={a2:.4f} (fark={a1-a2:+.4f})")
    if a1 - a2 > 0.01: print("  [!] overtraining belirtisi")
    return m, (Xtr, Xte, ytr, yte, wtr, wte)


def correlation_plot(X, title):
    c = X.corr(method="spearman")
    fig, ax = plt.subplots(figsize=(0.42*len(c)+2.5, 0.42*len(c)+2))
    im = ax.imshow(c, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(c))); ax.set_xticklabels(c.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(c))); ax.set_yticklabels(c.columns, fontsize=7)
    plt.colorbar(im, fraction=0.046); ax.set_title(title, fontsize=9); plt.show()
    hi = [(c.columns[i], c.columns[j], round(c.iloc[i,j],3))
          for i in range(len(c)) for j in range(i+1, len(c)) if abs(c.iloc[i,j]) > 0.85]
    for a,b,v in hi: print(f"  |rho|>0.85: {a:26s} {b:26s} {v:+.3f}")


def importance_plot(model, X, y, w, title, n_repeats=5):
    gain  = pd.Series(model.feature_importance("gain"),  index=X.columns)
    split = pd.Series(model.feature_importance("split"), index=X.columns)
    base = roc_auc_score(y, model.predict(X), sample_weight=w)
    perm = {}
    for col in X.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy(); Xp[col] = rng.permutation(Xp[col].to_numpy())
            drops.append(base - roc_auc_score(y, model.predict(Xp), sample_weight=w))
        perm[col] = float(np.mean(drops))
    imp = pd.DataFrame({"gain_%": 100*gain/max(gain.sum(),1),
                        "split_%": 100*split/max(split.sum(),1),
                        "perm_dAUC": pd.Series(perm)}).sort_values("perm_dAUC", ascending=False)
    fig, axes = plt.subplots(1, 3, figsize=(11, 0.28*len(imp)+2.2), sharey=True)
    for ax, col, c in zip(axes, ["gain_%","split_%","perm_dAUC"],
                          ["tab:blue","tab:orange","tab:green"]):
        ax.barh(imp.index[::-1], imp[col][::-1], color=c)
        ax.set_xlabel(col, fontsize=8); ax.tick_params(labelsize=7)
    fig.suptitle(title, fontsize=10); plt.tight_layout(); plt.show()
    return imp

In [ ]:
# Sinyal = mevcut tum GENIE tatlari (bu uretimde nutau YOK)
NU = [s for s in samples if SAMPLES[s]["kind"] == "genie"]
print("Sinyal ornekleri:", NU)
if "nutau" not in NU:
    print("  [i] nutau seti yok -> sinyal nue + numu.  Toplam sinyalin ~%3'u")
    print("      eksik; nutau'ya ozgu davranis varsa siniflandirici gormeyecek.")

# --- NOISE eğitim seti ---
if "noise" in samples and NU:
    noise_names = [f.name for f in NOISE_FEATURES]
    df_noise, X_noise, y_noise, w_noise = build_training_set(
        [samples[s] for s in NU], [samples["noise"]], noise_names)
    print("NOISE:", X_noise.shape, "sig:", int(y_noise.sum()), "bkg:", int((1-y_noise).sum()))
    correlation_plot(X_noise, "Noise BDT — Spearman")
    m_noise, sp_noise = quick_train(X_noise, y_noise, w_noise, LGB_NOISE, TRAIN_FRAC_NOISE)
    imp_noise = importance_plot(m_noise, sp_noise[1], sp_noise[3], sp_noise[5],
                                "Noise BDT — feature importance")
    display(imp_noise)

In [ ]:
# --- MUON eğitim seti: arka plan = NOISE KESIMI GECMIS gercek veri ---
if MUON_BACKGROUND in samples and NU:
    muon_names = [f.name for f in MUON_FEATURES]
    bkg = samples[MUON_BACKGROUND]
    print(f"Muon arka plani: {MUON_BACKGROUND} ({len(bkg):,} olay)")
    if MUON_BACKGROUND == "data" and MUON_TRAINING_RUNS:
        bkg = bkg[bkg["Run"].isin(MUON_TRAINING_RUNS)]
        print(f"  {bkg['Run'].nunique()} run'a filtrelendi")
    mask = after_noise_cut(bkg)
    print(f"Noise kesimi: {len(bkg):,} -> {int(mask.sum()):,} olay")
    bkg = bkg[mask]

    sig = [samples[s] for s in NU]
    sig = [s[after_noise_cut(s)] for s in sig]      # sinyale de ayni kesim

    df_muon, X_muon, y_muon, w_muon = build_training_set(sig, [bkg], muon_names)
    print("MUON:", X_muon.shape, "sig:", int(y_muon.sum()), "bkg:", int((1-y_muon).sum()))
    correlation_plot(X_muon, "Muon BDT — Spearman")
    m_muon, sp_muon = quick_train(X_muon, y_muon, w_muon, LGB_MUON, TRAIN_FRAC_MUON)
    imp_muon = importance_plot(m_muon, sp_muon[1], sp_muon[3], sp_muon[5],
                               "Muon BDT — feature importance")
    display(imp_muon)

In [ ]:
# --- Genisletilmis tarama: "~40 degisken test edildi" adiminin karsiligi ---
if "df_muon" in dir():
    cand = ([f.name for f in MUON_FEATURES] + [f.name for f in CANDIDATE_FEATURES]
            + DERIVED_NAMES)
    bad = set(sc[sc.mean_abs_dev > 0.25].feature)     # data/MC kotu olanlari at
    avail = [c for c in dict.fromkeys(cand) if c in df_muon.columns and c not in bad]
    print(f"{len(avail)} degisken taraniyor ({len(bad)} tanesi data/MC nedeniyle elendi)")
    Xe = df_muon[avail].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan)
    m_ext, sp_ext = quick_train(Xe, y_muon, w_muon, LGB_MUON, TRAIN_FRAC_MUON)
    imp_ext = importance_plot(m_ext, sp_ext[1], sp_ext[3], sp_ext[5],
                              "Genisletilmis muon taramasi", n_repeats=3)
    display(imp_ext.head(25))

In [ ]:
def incremental_scan(X_full, y, w, params, order, train_frac=0.5, max_n=20):
    res = []
    for n in range(1, min(max_n, len(order))+1):
        cols = order[:n]
        Xtr, Xte, ytr, yte, wtr, wte = train_test_split(
            X_full[cols], y, w, train_size=train_frac, random_state=RNG_SEED, stratify=y)
        d1 = lgb.Dataset(Xtr, ytr, weight=wtr)
        d2 = lgb.Dataset(Xte, yte, weight=wte, reference=d1)
        m = lgb.train(params, d1, 500, valid_sets=[d2],
                      callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(0)])
        res.append(dict(n=n, added=cols[-1],
                        auc=roc_auc_score(yte, m.predict(Xte), sample_weight=wte)))
    return pd.DataFrame(res)

if "imp_ext" in dir():
    scan = incremental_scan(Xe, y_muon, w_muon, LGB_MUON, list(imp_ext.index),
                            TRAIN_FRAC_MUON, max_n=20)
    display(scan)
    plt.figure(figsize=(6,3)); plt.plot(scan.n, scan.auc, "o-")
    plt.xlabel("degisken sayisi"); plt.ylabel("test AUC (agirlikli)")
    plt.grid(alpha=0.3); plt.title("Kac degisken yeterli?"); plt.show()

## 11. Export

Nihai değişken listesini üç kritere göre seçin:

1. Sağlık kontrolünden **OK** (bölüm 5)
2. Yeniden yazılan değişkenler doğrulandı (bölüm 6)
3. Data/MC sapması makul (bölüm 9) ve permutation importance anlamlı (bölüm 10)

Meta dosyası, işlemede kullanılan script sürümünü ve anahtar isimlerini de
kaydeder — ileride "bu modeli hangi kodla ürettim" sorusunu cevaplayabilmek için.

In [ ]:
FINAL_NOISE = [f.name for f in NOISE_FEATURES]
FINAL_MUON  = [f.name for f in MUON_FEATURES]

def code_fingerprint():
    import hashlib
    fp = {}
    for p in (VARS_PY, PROCESS_PY):
        if os.path.exists(p):
            fp[os.path.basename(p)] = hashlib.sha256(open(p,"rb").read()).hexdigest()[:16]
    return fp


def export(df, X, y, w, features, tag, train_frac):
    idx = np.arange(len(df))
    itr, _ = train_test_split(idx, train_size=train_frac,
                              random_state=RNG_SEED, stratify=y)
    split = np.array(["test"]*len(df), dtype=object); split[itr] = "train"

    out = X[features].copy()
    out["label"] = y; out["w_train"] = w
    out["w_phys"] = df["w_phys"].to_numpy(); out["split"] = split
    out["sample"] = df["sample"].to_numpy()
    for c in IDX:
        if c in df.columns: out[c] = df[c].to_numpy()

    path = os.path.join(OUT, f"L4_{tag}_training.parquet")
    out.to_parquet(path, index=False)

    meta = dict(tag=tag, features=features,
                n_events=int(len(out)), n_signal=int(y.sum()),
                n_background=int((1-y).sum()), train_frac=train_frac, seed=RNG_SEED,
                lgb_params=(LGB_NOISE if tag=="noise" else LGB_MUON),
                frame_keys=dict(micro_count_subkey=MICRO_SUBKEY,
                                hitstat=KEY_HITSTAT, vich_nch=KEY_VICH_NCH,
                                vich_speed_window=list(VICH_SPEED),
                                fill_ratio_radius=FILL_RADIUS),
                data_livetime_s=DATA_LIVETIME_S,
                muon_training_runs=(MUON_TRAINING_RUNS if tag=="muon" else None),
                code_sha256=code_fingerprint(),
                source_globs={k: v["hdf_glob"] for k, v in SAMPLES.items()})
    with open(os.path.join(OUT, f"L4_{tag}_meta.json"), "w") as fh:
        json.dump(meta, fh, indent=2, default=str)

    print(f"[{tag}] -> {path}")
    print(f"        {len(out):,} olay, {len(features)} degisken, "
          f"train={int((split=='train').sum()):,} test={int((split=='test').sum()):,}")
    return out

if "X_noise" in dir():
    export(df_noise, X_noise, y_noise, w_noise, FINAL_NOISE, "noise", TRAIN_FRAC_NOISE)
if "X_muon" in dir():
    export(df_muon, X_muon, y_muon, w_muon, FINAL_MUON, "muon", TRAIN_FRAC_MUON)

In [ ]:
print("Egitim komutlari:\n")
for tag in ("noise", "muon"):
    pq = os.path.join(OUT, f"L4_{tag}_training.parquet")
    mark = "" if os.path.exists(pq) else "   # <- parquet henuz yok"
    print(f"python {os.path.join(L4_CODE_DIR,'train_L4_classifier.py')} "
          f"--tag {tag} \\\n    --input {pq} \\\n    --outdir {MODEL_DIR}{mark}\n")

print("Sonra IceTray'de kesimi uygulamak icin:")
print(f"python {os.path.join(L4_CODE_DIR,'process_L4.py')} --gcd ... --input ... \\")
print(f"    --output-hdf5 ... --apply-cut --model-dir {MODEL_DIR}")

### Kontrol listesi

**İşleme (bölüm 1–2)**
- [ ] `run_smoke_test()` hatasız çalıştı
- [ ] `verify_booking()` beklenen tüm tabloları `OK` gösteriyor
- [ ] `--sub-event-stream` doğru (boş HDF5 gelmedi)
- [ ] Tüm örnekler için üretim job'ları tamamlandı

**Değişkenler (bölüm 5–6)**
- [ ] Sağlık raporunda BDT girdileri hepsi `OK`
- [ ] `VICH_nch` MuonGun'da nötrinolardan belirgin şekilde yüksek
- [ ] Referans dosya varsa `compare_to_reference()` çalıştırıldı
- [ ] `accumulated_time` ve `first_hlc_rho` dağılımları fiziksel görünüyor

**Ağırlıklar (bölüm 7)**
- [ ] `DATA_LIVETIME_S` makul (gün mertebesinde)
- [ ] MC rate'leri beklenen büyüklükte (gürültü ~36.6 mHz L3 sonrası)
- [ ] Kısmi job çalıştırılmadı, ya da `n_files` doğru

**Eğitim (bölüm 9–11)**
- [ ] Data/MC sapması yüksek değişkenler elendi veya nedeni anlaşıldı
- [ ] AUC train/test farkı < 0.01
- [ ] Muon arka planı gerçekten noise kesiminden geçti
- [ ] Eğitim run'larının analiz örneğinde kalıp kalmayacağına karar verildi

Sonraki adım: eğitim script'i — parquet'leri okuyup `I3Classifier`'ın beklediği
`.joblib` formatını üretecek. Önce `oscNext/tools/classifier.py` kaynağını açıp
model dosyasının içinde tam olarak neyi beklediğini doğrulamak gerekiyor.